In [1]:
import ibis
from utils.f_0_dirs import get_data_dirs

dirs = get_data_dirs("descriptives")
con = ibis.duckdb.connect(str(dirs.db_path))

In [13]:
from ibis import _, selectors as s
import pandas as pd

table_name = "working_yearly"
old_table_name = "fame_yearly_kp"

t_panel = con.table(table_name)
t_old = (
    con.table(old_table_name)
    .select("registered_number", "year", "tangibles")
)

t_filtered = (
    t_panel
    .left_join(t_old, ["registered_number", "year"])
    .filter(
        (_.gva1 > 0) &
        (_.total_assets > 0) &
        (_.fixed_total > 0) &
        (_.tangibles > 0)
    )
)
df_stats_wide = (
    t_filtered
    .mutate(
        total_y=_.total_assets / _.gva1,
        fixed_y=_.fixed_total / _.gva1,
        tangibles_y=_.tangibles / _.gva1
    )
    .select('total_y', 'fixed_y', 'tangibles_y')
    .aggregate(
        s.across(s.all(), { "mean": _.mean(), "std": _.std() })
    )
    .execute()
)

s_flat = df_stats_wide.iloc[0]
s_flat.index = pd.MultiIndex.from_tuples(
    [col.rsplit("_", 1) for col in s_flat.index], 
    names=["column", "stat"]
)
df_stats = s_flat.unstack(level="stat")[["mean", "std"]]
# Add a row for the number of observations
df_stats.loc["n_obs", "mean"] = t_filtered.count().execute()
display(df_stats)

stat,mean,std
column,,
fixed_y,5.995464,1399.493418
tangibles_y,3.534172,1309.000736
total_y,11.563473,1810.382383
n_obs,991367.000000,NaN
